# UC CosMx: WTX Panel Processing (Baseline)

QC filter, normalize (ComBat by slide), and AUCell annotate for 2 UC WTX samples (CD_B4, CD_B5).
Outputs used as baseline for imputation comparison (Fig 2).

In [ ]:
# ── Paths ──
DATA_DIR   = "/path/to/cosmx_data/uc_1k_6k_wtx/whole_trans"
OUTPUT_DIR = DATA_DIR + "/Processed_merged"

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import squidpy as sq
import matplotlib.pyplot as plt
import seaborn as sns
import os
import scanpy.external as sce

In [ ]:
sample_dir = '/path/to/cosmx_data/cd_6k_wtx/whole_trans/3 Raw data/R6221_CD_B4/'

In [ ]:
fov_file = [item for item in os.listdir(sample_dir) if 'fov_positions_file' in item][0]
fov_df = pd.read_csv(os.path.join(sample_dir, fov_file))
if 'FOV' in fov_df.columns:
  print("Refactoring file to older format.")
  # Rename 'FOV' column to 'fov'
  fov_df.rename(columns={'FOV': 'fov'}, inplace=True)
  # have fov_file reference the new, formatted file and write it
  fov_file = os.path.join(sample_dir,'fov_positions_formatted.csv')
  fov_df.to_csv(fov_file, index=False)

In [ ]:
adata = sq.read.nanostring(
    path=sample_dir,
    counts_file="wtx_CD_B4_exprMat_file.csv.gz",
    meta_file="wtx_CD_B4_metadata_file.csv.gz",
    fov_file="fov_positions_formatted.csv",
)

In [ ]:
category_column = "fov"
coords = adata.obsm["spatial_fov"]
fov_series = adata.obs[category_column].astype(str)

# --- build a numeric sort key from any digits in the label (e.g., "FOV_12" -> 12) ---
num_key = pd.to_numeric(fov_series.str.extract(r'(\d+)')[0], errors="coerce")

# unique labels + numeric keys -> sort by number, then by label; non-numeric go last
uniq = pd.DataFrame({"label": fov_series.unique()})
uniq["num"] = pd.to_numeric(uniq["label"].str.extract(r'(\d+)')[0], errors="coerce")
uniq = uniq.sort_values(["num", "label"], na_position="last")
ordered_labels = uniq["label"].tolist()

# Categorical with ordered categories
cats = pd.Categorical(fov_series, categories=ordered_labels, ordered=True)
codes = cats.codes

# Palette and colors (tab20 cycles; expand if >20 categories)
palette = sns.color_palette("tab20", len(ordered_labels))
point_colors = [palette[c] if c >= 0 else (0.7, 0.7, 0.7) for c in codes]

# --- plot ---
plt.figure(figsize=(9, 10))
plt.scatter(coords[:, 0], coords[:, 1], c=point_colors, s=1)
plt.title("CD_B4 Spatial Plot for FOVs")
plt.xlabel("X coordinate")
plt.ylabel("Y coordinate")

# Legend in numeric order
handles = [
    plt.Line2D([0], [0], marker="o", color="w",
               markerfacecolor=palette[i], markersize=8, linestyle="None")
    for i in range(len(ordered_labels))
]
ncols = 3  # choose how many columns you want

plt.legend(
    handles, ordered_labels,
    title=category_column,
    loc="upper left",
    ncol=ncols,
    frameon=False,
    columnspacing=1.0,
    handletextpad=0.4,
    prop={"size": 8},
    title_fontsize=9,
    bbox_to_anchor=(1.02, 1))

plt.tight_layout()
plt.show()


In [ ]:
sample_dir = '/path/to/cosmx_data/cd_6k_wtx/whole_trans/3 Raw data/R6221_CD_B5/'

In [ ]:
fov_file = [item for item in os.listdir(sample_dir) if 'fov_positions_file' in item][0]
fov_df = pd.read_csv(os.path.join(sample_dir, fov_file))
if 'FOV' in fov_df.columns:
  print("Refactoring file to older format.")
  # Rename 'FOV' column to 'fov'
  fov_df.rename(columns={'FOV': 'fov'}, inplace=True)
  # have fov_file reference the new, formatted file and write it
  fov_file = os.path.join(sample_dir,'fov_positions_formatted.csv')
  fov_df.to_csv(fov_file, index=False)

In [ ]:
adata2 = sq.read.nanostring(
    path=sample_dir,
    counts_file="wtx_CD_B5_exprMat_file.csv.gz",
    meta_file="wtx_CD_B5_metadata_file.csv.gz",
    fov_file="fov_positions_formatted.csv",
)


In [ ]:
category_column = "fov"
coords = adata2.obsm["spatial_fov"]
fov_series = adata2.obs[category_column].astype(str)

# --- build a numeric sort key from any digits in the label (e.g., "FOV_12" -> 12) ---
num_key = pd.to_numeric(fov_series.str.extract(r'(\d+)')[0], errors="coerce")

# unique labels + numeric keys -> sort by number, then by label; non-numeric go last
uniq = pd.DataFrame({"label": fov_series.unique()})
uniq["num"] = pd.to_numeric(uniq["label"].str.extract(r'(\d+)')[0], errors="coerce")
uniq = uniq.sort_values(["num", "label"], na_position="last")
ordered_labels = uniq["label"].tolist()

# Categorical with ordered categories
cats = pd.Categorical(fov_series, categories=ordered_labels, ordered=True)
codes = cats.codes

# Palette and colors (tab20 cycles; expand if >20 categories)
palette = sns.color_palette("tab20", len(ordered_labels))
point_colors = [palette[c] if c >= 0 else (0.7, 0.7, 0.7) for c in codes]

# --- plot ---
plt.figure(figsize=(9, 10))
plt.scatter(coords[:, 0], coords[:, 1], c=point_colors, s=1)
plt.title("CD_B5 Spatial Plot for FOVs")
plt.xlabel("X coordinate")
plt.ylabel("Y coordinate")

# Legend in numeric order
handles = [
    plt.Line2D([0], [0], marker="o", color="w",
               markerfacecolor=palette[i], markersize=8, linestyle="None")
    for i in range(len(ordered_labels))
]
ncols = 3  # choose how many columns you want

plt.legend(
    handles, ordered_labels,
    title=category_column,
    loc="upper left",
    ncol=ncols,
    frameon=False,
    columnspacing=1.0,
    handletextpad=0.4,
    prop={"size": 8},
    title_fontsize=9,
    bbox_to_anchor=(1.02, 1), 
)
plt.tight_layout()


plt.tight_layout()
plt.show()


In [ ]:
adata.var['features']=adata.var.index.tolist()
adata.obs[['orig.ident']]='CD_B4'
#adata.obs['patient']=['UC_P2' if int(i) <=31 else 'UC_P1' for i in adata.obs['fov'].tolist() ]
#adata.obs['condition']= 'Inflamed'
adata.obs['fov_cell_id']= [str(i)+'_'+str(j) for i,j in zip(adata.obs['fov'].tolist(), adata.obs['cell_id'].tolist())]

In [ ]:
adata2.var['features']=adata2.var.index.tolist()
adata2.obs[['orig.ident']]='CD_B5'
#adata.obs['patient']=['UC_P2' if int(i) <=31 else 'UC_P1' for i in adata.obs['fov'].tolist() ]
#adata.obs['condition']= 'Inflamed'
adata2.obs['fov_cell_id']= [str(i)+'_'+str(j) for i,j in zip(adata2.obs['fov'].tolist(), adata2.obs['cell_id'].tolist())]

In [ ]:
adata.layers['raw']=adata.X

In [ ]:
adata2.layers['raw']=adata2.X

In [ ]:
slide=[]
slide1_list = [1, 2, 3, 4, 5, 6, 7, 10, 11, 12, 13, 17, 18, 19, 20, 23, 24, 25, 26, 29, 30, 31]

slide2_list = [8, 9, 14, 15, 16, 21, 22, 27, 28, 32, 33, 34]

slide3_list = [37, 38, 43, 44, 50, 51, 57, 58, 64, 65, 70, 71, 77, 78, 79]

slide4_list = [35, 36, 39, 40, 41, 42, 45, 46, 47, 52, 53, 54, 59, 60, 61, 66, 67, 72, 73]

for i,j in zip(adata.obs['orig.ident'].tolist(),adata.obs['fov'].tolist()):
    j= int(j)
    if j in slide1_list:
        slide.append(i+'_'+'slide1')
    elif j in slide2_list:
        slide.append(i+'_'+'slide2')
    elif j in slide3_list:
        slide.append(i+'_'+'slide3')
    elif j in slide4_list:
        slide.append(i+'_'+'slide4')
    else:
        slide.append(i+'_'+'slide5')

In [ ]:
adata.obs['slide']=slide

In [ ]:
adata_clean = adata[~adata.obs['fov'].isin([8,54,55])]

In [ ]:
category_column = "fov"
coords = adata_clean.obsm["spatial_fov"]
fov_series = adata_clean.obs[category_column].astype(str)

# --- build a numeric sort key from any digits in the label (e.g., "FOV_12" -> 12) ---
num_key = pd.to_numeric(fov_series.str.extract(r'(\d+)')[0], errors="coerce")

# unique labels + numeric keys -> sort by number, then by label; non-numeric go last
uniq = pd.DataFrame({"label": fov_series.unique()})
uniq["num"] = pd.to_numeric(uniq["label"].str.extract(r'(\d+)')[0], errors="coerce")
uniq = uniq.sort_values(["num", "label"], na_position="last")
ordered_labels = uniq["label"].tolist()

# Categorical with ordered categories
cats = pd.Categorical(fov_series, categories=ordered_labels, ordered=True)
codes = cats.codes

# Palette and colors (tab20 cycles; expand if >20 categories)
palette = sns.color_palette("tab20", len(ordered_labels))
point_colors = [palette[c] if c >= 0 else (0.7, 0.7, 0.7) for c in codes]

# --- plot ---
plt.figure(figsize=(9, 10))
plt.scatter(coords[:, 0], coords[:, 1], c=point_colors, s=1)
plt.title("CD_B4 Clean Spatial Plot for FOVs")
plt.xlabel("X coordinate")
plt.ylabel("Y coordinate")

# Legend in numeric order
handles = [
    plt.Line2D([0], [0], marker="o", color="w",
               markerfacecolor=palette[i], markersize=8, linestyle="None")
    for i in range(len(ordered_labels))
]
ncols = 3  # choose how many columns you want

plt.legend(
    handles, ordered_labels,
    title=category_column,
    loc="upper left",
    ncol=ncols,
    frameon=False,
    columnspacing=1.0,
    handletextpad=0.4,
    prop={"size": 8},
    title_fontsize=9,
    bbox_to_anchor=(1.02, 1), 
)
plt.tight_layout()


plt.tight_layout()
plt.show()


In [ ]:
slide=[]
slide1_list = list(range(17))                      # [0, 1, 2, ..., 16]
slide2_list = [17, 18, 19, 20, 21]
slide3_list = list(range(22, 51)) + [70]           # [22..50] + 70
slide4_list = [51, 52, 53, 69]
slide5_list = list(range(54, 71))                  # [54..70]

for i,j in zip(adata2.obs['orig.ident'].tolist(),adata2.obs['fov'].tolist()):
    j= int(j)
    if j in slide1_list:
        slide.append(i+'_'+'slide1')
    elif j in slide2_list:
        slide.append(i+'_'+'slide2')
    elif j in slide3_list:
        slide.append(i+'_'+'slide3')
    elif j in slide4_list:
        slide.append(i+'_'+'slide4')
    elif j in slide5_list:
        slide.append(i+'_'+'slide5')
    else:
        print(j,'error')

In [ ]:
adata2.obs['slide']=slide

In [ ]:
adata_clean2 = adata2[~adata2.obs['fov'].isin([15,16,21,22,25,29,36,40,54,55])]

In [ ]:
category_column = "fov"
coords = adata_clean2.obsm["spatial_fov"]
fov_series = adata_clean2.obs[category_column].astype(str)

# --- build a numeric sort key from any digits in the label (e.g., "FOV_12" -> 12) ---
num_key = pd.to_numeric(fov_series.str.extract(r'(\d+)')[0], errors="coerce")

# unique labels + numeric keys -> sort by number, then by label; non-numeric go last
uniq = pd.DataFrame({"label": fov_series.unique()})
uniq["num"] = pd.to_numeric(uniq["label"].str.extract(r'(\d+)')[0], errors="coerce")
uniq = uniq.sort_values(["num", "label"], na_position="last")
ordered_labels = uniq["label"].tolist()

# Categorical with ordered categories
cats = pd.Categorical(fov_series, categories=ordered_labels, ordered=True)
codes = cats.codes

# Palette and colors (tab20 cycles; expand if >20 categories)
palette = sns.color_palette("tab20", len(ordered_labels))
point_colors = [palette[c] if c >= 0 else (0.7, 0.7, 0.7) for c in codes]

# --- plot ---
plt.figure(figsize=(9, 10))
plt.scatter(coords[:, 0], coords[:, 1], c=point_colors, s=1)
plt.title("CD_B5 Clean Spatial Plot for FOVs")
plt.xlabel("X coordinate")
plt.ylabel("Y coordinate")

# Legend in numeric order
handles = [
    plt.Line2D([0], [0], marker="o", color="w",
               markerfacecolor=palette[i], markersize=8, linestyle="None")
    for i in range(len(ordered_labels))
]
ncols = 3  # choose how many columns you want

plt.legend(
    handles, ordered_labels,
    title=category_column,
    loc="upper left",
    ncol=ncols,
    frameon=False,
    columnspacing=1.0,
    handletextpad=0.4,
    prop={"size": 8},
    title_fontsize=9,
    bbox_to_anchor=(1.02, 1), 
)
plt.tight_layout()


plt.tight_layout()
plt.show()


In [ ]:
adata.var['neg_probe']=[True if 'Negative' in i else False for i in adata.var.index.tolist()]
adata.var['control_probe']=[True if 'SystemContro' in i else False for i in adata.var.index.tolist()]
adata.var

In [ ]:
adata2.var['neg_probe']=[True if 'Negative' in i else False for i in adata2.var.index.tolist()]
adata2.var['control_probe']=[True if 'SystemContro' in i else False for i in adata2.var.index.tolist()]
adata2.var

In [ ]:
adata=adata[:, ~adata.var["neg_probe"]]
adata=adata[:, ~adata.var["control_probe"]]
adata2=adata2[:, ~adata2.var["neg_probe"]]
adata2=adata2[:, ~adata2.var["control_probe"]]

In [ ]:
sc.pp.calculate_qc_metrics(
    adata, inplace=True, log1p=True
)
sc.pp.calculate_qc_metrics(
    adata2, inplace=True, log1p=True
)

In [ ]:
sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts",'Area'],
    jitter=0.4,
    multi_panel=True,
)

In [ ]:
sc.pl.violin(
    adata2,
    ["n_genes_by_counts", "total_counts",'Area'],
    jitter=0.4,
    multi_panel=True,
)

In [ ]:
# only remove cells that have too little gene counts
sc.pp.filter_cells(adata, min_counts=400)
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_cells(adata, max_counts=10000)
sc.pp.filter_cells(adata, max_genes=6000)

In [ ]:
adata=adata[adata.obs['Area']<=40000]

In [ ]:
# only remove cells that have too little gene counts
sc.pp.filter_cells(adata2, min_counts=400)
sc.pp.filter_cells(adata2, min_genes=200)
sc.pp.filter_cells(adata2, max_counts=10000)
sc.pp.filter_cells(adata2, max_genes=6000)

In [ ]:
adata2=adata2[adata2.obs['Area']<=40000]

In [ ]:
sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts",'Area'],
    jitter=0.4,
    multi_panel=True,
)

In [ ]:
sc.pl.violin(
    adata2,
    ["n_genes_by_counts", "total_counts",'Area'],
    jitter=0.4,
    multi_panel=True,
)

In [ ]:
adata_cosmx=sc.concat([adata, adata2])

In [ ]:
sc.pp.normalize_total(adata_cosmx, inplace=True)
sc.pp.log1p(adata_cosmx)

In [ ]:
adata.write_h5ad('/path/to/cosmx_data/cd_6k_wtx/whole_trans/Processed_CD_B4/h5ad_obj/qc.h5ad')
adata2.write_h5ad('/path/to/cosmx_data/cd_6k_wtx/whole_trans/Processed_CD_B5/h5ad_obj/qc.h5ad')

In [ ]:
adata_cosmx.write_h5ad('/path/to/cosmx_data/cd_6k_wtx/whole_trans/Processed_merged/h5ad_obj/qc_norm.h5ad')

In [ ]:
norm_log = adata_cosmx.X.copy()

In [ ]:
sc.pp.combat(adata_cosmx, key="slide") 

harmony kept on crashing

In [ ]:
sc.tl.pca(adata_cosmx, n_comps=50, svd_solver="arpack")  # avoids full dense mem usage


In [ ]:
adata_cosmx.layers['combat']=adata_cosmx.X

In [ ]:
adata_cosmx.X=norm_log

In [ ]:
adata_cosmx.write_h5ad('/path/to/cosmx_data/cd_6k_wtx/whole_trans/Processed_merged/h5ad_obj/qc_norm_umap.h5ad')

In [ ]:
adata_cosmx1=adata_cosmx[adata_cosmx.obs['orig.ident']=='CD_B4']
adata_cosmx2=adata_cosmx[adata_cosmx.obs['orig.ident']=='CD_B5']

In [ ]:
adata_cosmx1.write_h5ad('/path/to/cosmx_data/cd_6k_wtx/whole_trans/Processed_CD_B4/h5ad_obj/qc.h5ad')
adata_cosmx2.write_h5ad('/path/to/cosmx_data/cd_6k_wtx/whole_trans/Processed_CD_B5/h5ad_obj/qc.h5ad')

In [ ]:
adata_cosmx_norm_nolog1p = adata_cosmx.copy()
adata_cosmx_norm_nolog1p.X = adata_cosmx_norm_nolog1p.layers['raw']


In [ ]:
sc.pp.normalize_total(adata_cosmx_norm_nolog1p, inplace=True)

In [ ]:
adata_cosmx_norm_nolog1p_count = pd.DataFrame(adata_cosmx_norm_nolog1p.X.toarray())

In [ ]:
adata_cosmx_norm_nolog1p_count.columns = adata_cosmx_norm_nolog1p.var.index.tolist()
adata_cosmx_norm_nolog1p_count.index=adata_cosmx_norm_nolog1p.obs['cell'].tolist()

In [ ]:
adata_cosmx_norm_nolog1p_count.to_csv('/path/to/cosmx_data/cd_6k_wtx/whole_trans/Processed_merged/count_csv/norm_counts.csv')

# anno

In [ ]:
anno = pd.read_csv('/path/to/cosmx_data/cd_6k_wtx/whole_trans/Processed_merged/anno/norm_counts_aucell.csv')


In [ ]:
anno['label'].value_counts()

In [ ]:
mapper= pd.read_csv('/path/to/scrna/cd/cell_type_category_map.csv')

In [ ]:
# Step 1: Clean the names
mapper['cell_type_aucell'] = mapper['cell type'].str.replace(r'[+\-\/()]', ' ', regex=True)
mapper['cell_type_aucell'] = mapper['cell_type_aucell'].str.replace(r'\s+', ' ', regex=True).str.strip()

# Step 2: Disambiguate duplicates by appending " 1", " 2", etc.
mapper['cell_type_aucell'] = (
    mapper.groupby('cell_type_aucell').cumcount().astype(str).replace('0', '', regex=False)
    .radd(mapper['cell_type_aucell'] + ' ').str.strip()
)

In [ ]:
anno['label_clean'] = anno['label'].str.rstrip()

In [ ]:
anno_map=anno.merge(mapper, how = 'left', left_on = 'label_clean',right_on = 'cell_type_aucell')

In [ ]:
adata_cosmx.obs['cell_type'] = anno_map['cell type'].tolist()
adata_cosmx.obs['cell_type_short'] = anno_map['cell type short'].tolist()
adata_cosmx.obs['cell_category'] = anno_map['cell category'].tolist()
adata_cosmx.obs['cell_type_general'] = anno_map['cell type general'].tolist()

In [ ]:
original_cats = list(adata_cosmx.obs['cell_category'].astype('category').cat.categories)
reordered_cats = [cat for cat in original_cats if cat != 'Cycling'] + ['Cycling']

adata_cosmx.obs['cell_category'] = pd.Categorical(
    adata_cosmx.obs['cell_category'],
    categories=reordered_cats,
    ordered=True
)

In [ ]:
adata_cosmx1=adata_cosmx[adata_cosmx.obs['orig.ident']=='CD_B4']
adata_cosmx2=adata_cosmx[adata_cosmx.obs['orig.ident']=='CD_B5']

In [ ]:
# Create a color palette
coordinates = adata_cosmx2.obsm['spatial_fov']

category_column = 'cell_category'
categories = adata_cosmx2.obs[category_column].astype('category')

palette = sns.color_palette('tab20', len(categories.cat.categories))
colors = categories.cat.codes.map(dict(enumerate(palette)))

# Plotting
plt.figure(figsize=(9, 10))
scatter = plt.scatter(coordinates[:, 0], coordinates[:, 1], c=colors, s=0.5)  # Adjust `s` for marker size
plt.title("CD_B5 Spatial Plot for Cell Category")
plt.xlabel("X coordinate")
plt.ylabel("Y coordinate")

# Create a custom legend
handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=palette[i], markersize=8) 
           for i in range(len(categories.cat.categories))]
plt.legend(handles, categories.cat.categories, title=category_column, bbox_to_anchor=(1.05, 1), loc='upper left')

plt.show()

In [ ]:
adata_cosmx2_s1 = adata_cosmx2[adata_cosmx2.obs['slide']=='CD_B5_slide1']

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

coordinates = adata_cosmx2_s1.obsm["spatial_fov"]
category_column = "cell_category"

# keep original categorical order if it exists
labels = adata_cosmx2_s1.obs[category_column]
if str(labels.dtype) == "category":
    categories = labels.cat.categories  # preserves existing order
else:
    categories = pd.unique(labels)      # fallback (in existing order)
labels = labels.astype("object").fillna("Other")

# build color palette matching category order
palette = sns.color_palette("tab20", n_colors=len(categories))
cat_to_color = dict(zip(categories, palette))

# map each label to its color (fallback = light gray for unexpected)
colors = labels.map(lambda k: cat_to_color.get(k, "#cccccc")).to_numpy()

# plot
plt.figure(figsize=(8, 10))
plt.scatter(
    coordinates[:, 0],
    coordinates[:, 1],
    c=colors,
    s=3,
    linewidths=0
)

plt.title("Inf CD_B5 Slide 1 Cell Categories", fontsize=16, pad=15)
plt.xlabel("")
plt.ylabel("")
plt.xticks([])
plt.yticks([])
plt.grid(False)

# legend – keep only categories actually present
cats_present = [c for c in categories if (labels == c).any()]
handles = [
    plt.Line2D([0], [0],
               marker='o', linestyle='None',
               markerfacecolor=cat_to_color[c],
               markeredgecolor='none',
               markersize=10)
    for c in cats_present
]
plt.legend(
    handles,
    cats_present,
    title="Cell Category",
    loc='lower left',
    frameon=False,
    fontsize=11,
    title_fontsize=12
)

# hide all spines
ax = plt.gca()
for spine in ax.spines.values():
    spine.set_visible(False)

plt.savefig("inf_CD_B4_s1_cell_categories.jpg", bbox_inches='tight', dpi=300)
plt.show()


In [ ]:
adata_cosmx2_s3 = adata_cosmx2[adata_cosmx2.obs['slide']=='CD_B5_slide3']
adata_cosmx2_s5 = adata_cosmx2[adata_cosmx2.obs['slide']=='CD_B5_slide5']

In [ ]:
# Create a color palette
coordinates = adata_cosmx2_s3.obsm['spatial_fov']

category_column = 'cell_category'
categories = adata_cosmx2_s3.obs[category_column].astype('category')

palette = sns.color_palette('tab20', len(categories.cat.categories))
colors = categories.cat.codes.map(dict(enumerate(palette)))

# Plotting
plt.figure(figsize=(9, 10))
scatter = plt.scatter(coordinates[:, 0], coordinates[:, 1], c=colors, s=2)  # Adjust `s` for marker size
plt.title("CD_B5 Slide 1 Spatial Plot for Cell Category")
plt.xlabel("X coordinate")
plt.ylabel("Y coordinate")

# Create a custom legend
handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=palette[i], markersize=8) 
           for i in range(len(categories.cat.categories))]
plt.legend(handles, categories.cat.categories, title=category_column, bbox_to_anchor=(1.05, 1), loc='upper left')

plt.show()

In [ ]:
# Create a color palette
coordinates = adata_cosmx2_s5.obsm['spatial_fov']

category_column = 'cell_category'
categories = adata_cosmx2_s5.obs[category_column].astype('category')

palette = sns.color_palette('tab20', len(categories.cat.categories))
colors = categories.cat.codes.map(dict(enumerate(palette)))

# Plotting
plt.figure(figsize=(9, 10))
scatter = plt.scatter(coordinates[:, 0], coordinates[:, 1], c=colors, s=2)  # Adjust `s` for marker size
plt.title("CD_B5 Slide 5 Spatial Plot for Cell Category")
plt.xlabel("X coordinate")
plt.ylabel("Y coordinate")

# Create a custom legend
handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=palette[i], markersize=8) 
           for i in range(len(categories.cat.categories))]
plt.legend(handles, categories.cat.categories, title=category_column, bbox_to_anchor=(1.05, 1), loc='upper left')

plt.show()

In [ ]:
# Create a color palette
coordinates = adata_cosmx1.obsm['spatial_fov']

category_column = 'cell_category'
categories = adata_cosmx1.obs[category_column].astype('category')

palette = sns.color_palette('tab20', len(categories.cat.categories))
colors = categories.cat.codes.map(dict(enumerate(palette)))

# Plotting
plt.figure(figsize=(9, 10))
scatter = plt.scatter(coordinates[:, 0], coordinates[:, 1], c=colors, s=0.5)  # Adjust `s` for marker size
plt.title("CD_B4 Spatial Plot for Cell Category")
plt.xlabel("X coordinate")
plt.ylabel("Y coordinate")

# Create a custom legend
handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=palette[i], markersize=8) 
           for i in range(len(categories.cat.categories))]
plt.legend(handles, categories.cat.categories, title=category_column, bbox_to_anchor=(1.05, 1), loc='upper left')

plt.show()

In [ ]:
adata_cosmx1_s1 = adata_cosmx1[adata_cosmx1.obs['slide']=='CD_B4_slide1']
adata_cosmx1_s3 = adata_cosmx1[adata_cosmx1.obs['slide']=='CD_B4_slide3']
adata_cosmx1_s4 = adata_cosmx1[adata_cosmx1.obs['slide']=='CD_B4_slide4']

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

coordinates = adata_cosmx1_s1.obsm["spatial_fov"]
value_column = "Max.CD68"  # continuous variable

# Extract values
values = adata_cosmx1_s1.obs[value_column].astype(float)

# Optionally clip outliers for better visualization
vmin, vmax = values.quantile([0.01, 0.99])  # or (0, 1)
values_clipped = values.clip(vmin, vmax)

# Define colormap
cmap = "magma"  # alternatives: "magma", "coolwarm", "plasma"

# Plot
plt.figure(figsize=(8, 10))
sc = plt.scatter(
    coordinates[:, 0],
    coordinates[:, 1],
    c=values_clipped,
    cmap=cmap,
    s=3,
    linewidths=0
)

plt.title("Inf CD_B4 Slide 1 – Max.CD68 Expression", fontsize=16, pad=15)
plt.xlabel("")
plt.ylabel("")
plt.xticks([])
plt.yticks([])
plt.grid(False)

# Colorbar
cbar = plt.colorbar(sc, fraction=0.046, pad=0.04)
cbar.set_label("Max.CD68", rotation=270, labelpad=15, fontsize=12)

# Hide spines
ax = plt.gca()
for spine in ax.spines.values():
    spine.set_visible(False)

plt.savefig("inf_CD_B4_s1_MaxCD68.jpg", bbox_inches='tight', dpi=300)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

coordinates = adata_cosmx1_s1.obsm["spatial_fov"]
value_column = "Max.PanCK"  # continuous variable

# Extract values
values = adata_cosmx1_s1.obs[value_column].astype(float)

# Optionally clip outliers for better visualization
vmin, vmax = values.quantile([0.01, 0.99])  # or (0, 1)
values_clipped = values.clip(vmin, vmax)

# Define colormap
cmap = "magma"  # alternatives: "magma", "coolwarm", "plasma"

# Plot
plt.figure(figsize=(8, 10))
sc = plt.scatter(
    coordinates[:, 0],
    coordinates[:, 1],
    c=values_clipped,
    cmap=cmap,
    s=3,
    linewidths=0
)

plt.title("Inf CD_B4 Slide 1 – Max.CD68 Expression", fontsize=16, pad=15)
plt.xlabel("")
plt.ylabel("")
plt.xticks([])
plt.yticks([])
plt.grid(False)

# Colorbar
cbar = plt.colorbar(sc, fraction=0.046, pad=0.04)
cbar.set_label("Max.PanCK", rotation=270, labelpad=15, fontsize=12)

# Hide spines
ax = plt.gca()
for spine in ax.spines.values():
    spine.set_visible(False)

plt.savefig("inf_CD_B4_s1_MaxCD68.jpg", bbox_inches='tight', dpi=300)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

coordinates = adata_cosmx1_s1.obsm["spatial_fov"]
gene = "IGKC"

# 1. Extract expression values for the gene
# adata[:, gene].X returns an n_cells × 1 matrix
values = adata_cosmx1_s1[:, gene].X

# Handle dense/sparse matrix conversion
if not isinstance(values, np.ndarray):
    values = values.toarray().flatten()
else:
    values = values.flatten()

# 2. Optionally clip outliers for nicer color scale
vmin, vmax = np.quantile(values, [0.01, 0.99])
values_clipped = np.clip(values, vmin, vmax)

# 3. Plot
plt.figure(figsize=(8, 10))
sc = plt.scatter(
    coordinates[:, 0],
    coordinates[:, 1],
    c=values_clipped,
    cmap="magma",  # or "magma", "plasma", "coolwarm"
    s=3,
    linewidths=0
)

plt.title(f"Inf CD_B4 Slide 1 – {gene} Expression", fontsize=16, pad=15)
plt.xlabel("")
plt.ylabel("")
plt.xticks([])
plt.yticks([])
plt.grid(False)

# 4. Colorbar
cbar = plt.colorbar(sc, fraction=0.046, pad=0.04)
cbar.set_label(f"{gene} expression", rotation=270, labelpad=15, fontsize=12)

# 5. Hide spines
ax = plt.gca()
for spine in ax.spines.values():
    spine.set_visible(False)

plt.savefig(f"inf_CD_B4_s1_{gene}_expression.jpg", bbox_inches='tight', dpi=300)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

coordinates = adata_cosmx1_s1.obsm["spatial_fov"]
gene = "IGHA1"

# 1. Extract expression values for the gene
# adata[:, gene].X returns an n_cells × 1 matrix
values = adata_cosmx1_s1[:, gene].X

# Handle dense/sparse matrix conversion
if not isinstance(values, np.ndarray):
    values = values.toarray().flatten()
else:
    values = values.flatten()

# 2. Optionally clip outliers for nicer color scale
vmin, vmax = np.quantile(values, [0.01, 0.99])
values_clipped = np.clip(values, vmin, vmax)

# 3. Plot
plt.figure(figsize=(8, 10))
sc = plt.scatter(
    coordinates[:, 0],
    coordinates[:, 1],
    c=values_clipped,
    cmap="magma",  # or "magma", "plasma", "coolwarm"
    s=3,
    linewidths=0
)

plt.title(f"Inf CD_B4 Slide 1 – {gene} Expression", fontsize=16, pad=15)
plt.xlabel("")
plt.ylabel("")
plt.xticks([])
plt.yticks([])
plt.grid(False)

# 4. Colorbar
cbar = plt.colorbar(sc, fraction=0.046, pad=0.04)
cbar.set_label(f"{gene} expression", rotation=270, labelpad=15, fontsize=12)

# 5. Hide spines
ax = plt.gca()
for spine in ax.spines.values():
    spine.set_visible(False)

plt.savefig(f"inf_CD_B4_s1_{gene}_expression.jpg", bbox_inches='tight', dpi=300)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

coordinates = adata_cosmx1_s1.obsm["spatial_fov"]
gene = "IGHG1"

# 1. Extract expression values for the gene
# adata[:, gene].X returns an n_cells × 1 matrix
values = adata_cosmx1_s1[:, gene].X

# Handle dense/sparse matrix conversion
if not isinstance(values, np.ndarray):
    values = values.toarray().flatten()
else:
    values = values.flatten()

# 2. Optionally clip outliers for nicer color scale
vmin, vmax = np.quantile(values, [0.01, 0.99])
values_clipped = np.clip(values, vmin, vmax)

# 3. Plot
plt.figure(figsize=(8, 10))
sc = plt.scatter(
    coordinates[:, 0],
    coordinates[:, 1],
    c=values_clipped,
    cmap="magma",  # or "magma", "plasma", "coolwarm"
    s=3,
    linewidths=0
)

plt.title(f"Inf CD_B4 Slide 1 – {gene} Expression", fontsize=16, pad=15)
plt.xlabel("")
plt.ylabel("")
plt.xticks([])
plt.yticks([])
plt.grid(False)

# 4. Colorbar
cbar = plt.colorbar(sc, fraction=0.046, pad=0.04)
cbar.set_label(f"{gene} expression", rotation=270, labelpad=15, fontsize=12)

# 5. Hide spines
ax = plt.gca()
for spine in ax.spines.values():
    spine.set_visible(False)

plt.savefig(f"inf_CD_B4_s1_{gene}_expression.jpg", bbox_inches='tight', dpi=300)
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

adata = adata_cosmx1_s1  # replace if needed
plasma_label = "Plasma"  # label used in adata.obs['cell_category'] or your annotation column
cat_col = "cell_category"  # change if different

# 1) Basic counts & fraction per sample (or FOV)
total = adata.n_obs
if cat_col in adata.obs:
    n_plasma = (adata.obs[cat_col] == plasma_label).sum()
    print(f"Plasma cells: {n_plasma} / {total} ({n_plasma/total:.2%})")

# 2) Key canonical plasma markers (adjust to your panel)
plasma_markers = ["PRDM1","IRF4","XBP1","SDC1","MZB1"]  # TFs/plasma structural
ig_genes = [g for g in ["IGKC","IGHG1","IGHG2","IGHM","IGHA1","IGHA2","IGLL5","IGLC2"] if g in adata.var_names]

print("Immunoglobulin genes present in panel:", ig_genes)

# 4) Fraction of annotated plasma cells expressing IG genes
if len(ig_genes) > 0 and cat_col in adata.obs:
    mask = adata.obs[cat_col] == plasma_label
    expr = adata[mask, ig_genes].X
    # handle sparse/dense
    if hasattr(expr, "toarray"):
        expr = expr.toarray()
    expr_bool = (expr > 0)
    perc_expr = expr_bool.sum(axis=0) / expr_bool.shape[0]
    print("Percent of annotated plasma cells expressing each Ig gene:")
    print(pd.Series(perc_expr.flatten(), index=ig_genes).sort_values(ascending=False))


In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import math

# define your gene list
markers_plot = ig_genes + ["PAX5", "MS4A1"]

# number of columns per row
ncols = 2
nrows = math.ceil(len(markers_plot) / ncols)

fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(10, 4 * nrows))
axes = axes.flatten()

for i, gene in enumerate(markers_plot):
    sc.pl.violin(
        adata,
        keys=gene,
        groupby=cat_col,
        rotation=90,
        stripplot=False,
        size=2,
        log=True,
        ax=axes[i],
        show=False
    )
    
# hide any empty subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

coordinates = adata_cosmx1_s1.obsm["spatial_fov"]
category_column = "cell_category"

# keep original categorical order if it exists
labels = adata_cosmx1_s1.obs[category_column]
if str(labels.dtype) == "category":
    categories = labels.cat.categories  # preserves existing order
else:
    categories = pd.unique(labels)      # fallback (in existing order)
labels = labels.astype("object").fillna("Other")

# build color palette matching category order
palette = sns.color_palette("tab20", n_colors=len(categories))
cat_to_color = dict(zip(categories, palette))

# map each label to its color (fallback = light gray for unexpected)
colors = labels.map(lambda k: cat_to_color.get(k, "#cccccc")).to_numpy()

# plot
plt.figure(figsize=(8, 10))
plt.scatter(
    coordinates[:, 0],
    coordinates[:, 1],
    c=colors,
    s=3,
    linewidths=0
)

plt.title("Inf CD_B4 Slide 1 Cell Categories", fontsize=16, pad=15)
plt.xlabel("")
plt.ylabel("")
plt.xticks([])
plt.yticks([])
plt.grid(False)

# legend – keep only categories actually present
cats_present = [c for c in categories if (labels == c).any()]
handles = [
    plt.Line2D([0], [0],
               marker='o', linestyle='None',
               markerfacecolor=cat_to_color[c],
               markeredgecolor='none',
               markersize=10)
    for c in cats_present
]
plt.legend(
    handles,
    cats_present,
    title="Cell Category",
    loc='lower left',
    frameon=False,
    fontsize=11,
    title_fontsize=12
)

# hide all spines
ax = plt.gca()
for spine in ax.spines.values():
    spine.set_visible(False)

plt.savefig("inf_CD_B4_s1_cell_categories.jpg", bbox_inches='tight', dpi=300)
plt.show()


In [ ]:
# Create a color palette
coordinates = adata_cosmx1_s3.obsm['spatial_fov']

category_column = 'cell_category'
categories = adata_cosmx1_s3.obs[category_column].astype('category')

palette = sns.color_palette('tab20', len(categories.cat.categories))
colors = categories.cat.codes.map(dict(enumerate(palette)))

# Plotting
plt.figure(figsize=(9, 10))
scatter = plt.scatter(coordinates[:, 0], coordinates[:, 1], c=colors, s=2)  # Adjust `s` for marker size
plt.title("CD_B4 Slide 3 Spatial Plot for Cell Category")
plt.xlabel("X coordinate")
plt.ylabel("Y coordinate")

# Create a custom legend
handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=palette[i], markersize=8) 
           for i in range(len(categories.cat.categories))]
plt.legend(handles, categories.cat.categories, title=category_column, bbox_to_anchor=(1.05, 1), loc='upper left')

plt.show()

In [ ]:
# Create a color palette
coordinates = adata_cosmx1_s4.obsm['spatial_fov']

category_column = 'cell_category'
categories = adata_cosmx1_s4.obs[category_column].astype('category')

palette = sns.color_palette('tab20', len(categories.cat.categories))
colors = categories.cat.codes.map(dict(enumerate(palette)))

# Plotting
plt.figure(figsize=(9, 10))
scatter = plt.scatter(coordinates[:, 0], coordinates[:, 1], c=colors, s=2)  # Adjust `s` for marker size
plt.title("CD_B4 Slide 4 Spatial Plot for Cell Category")
plt.xlabel("X coordinate")
plt.ylabel("Y coordinate")

# Create a custom legend
handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=palette[i], markersize=8) 
           for i in range(len(categories.cat.categories))]
plt.legend(handles, categories.cat.categories, title=category_column, bbox_to_anchor=(1.05, 1), loc='upper left')

plt.show()

In [ ]:
adata_cosmx.write_h5ad('/path/to/cosmx_data/cd_6k_wtx/whole_trans/Processed_merged/h5ad_obj/norm_anno.h5ad')

In [ ]:
adata_cosmx1.write_h5ad('/path/to/cosmx_data/cd_6k_wtx/whole_trans/Processed_CD_B4/h5ad_obj/norm_anno.h5ad')
adata_cosmx2.write_h5ad('/path/to/cosmx_data/cd_6k_wtx/whole_trans/Processed_CD_B5/h5ad_obj/norm_anno.h5ad')

In [ ]:
adata_cosmx.obs['slide'].unique()

In [ ]:
patient =[]
condition = []

for i,j in adata_cosmx.obs.iterrows():
    if j['orig.ident'] == 'CD_B4':
        if j['slide'] in ['CD_B4_slide1','CD_B4_slide2']:
            patient.append('CD_B4')
            condition.append('Inf')
        else:
            patient.append('UC_NI1')
            condition.append('Non-Inf')
    else:
        if j['slide'] in ['CD_B5_slide1','CD_B5_slide2']:
            patient.append('CD_B5')
            condition.append('Inf')
        else:
            patient.append('UC_NI2')
            condition.append('Healthy')

In [ ]:
adata_cosmx.obs['patient'] = patient
adata_cosmx.obs['condition'] = condition

In [ ]:
adata_cosmx1=adata_cosmx[adata_cosmx.obs['orig.ident']=='CD_B4']
adata_cosmx2=adata_cosmx[adata_cosmx.obs['orig.ident']=='CD_B5']

In [ ]:
# Create a color palette
coordinates = adata_cosmx1.obsm['spatial_fov']

category_column = 'patient'
categories = adata_cosmx1.obs[category_column].astype('category')

palette = sns.color_palette('tab20', len(categories.cat.categories))
colors = categories.cat.codes.map(dict(enumerate(palette)))

# Plotting
plt.figure(figsize=(9, 10))
scatter = plt.scatter(coordinates[:, 0], coordinates[:, 1], c=colors, s=0.5)  # Adjust `s` for marker size
plt.title("CD_B4 Spatial Plot for Patient")
plt.xlabel("X coordinate")
plt.ylabel("Y coordinate")

# Create a custom legend
handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=palette[i], markersize=8) 
           for i in range(len(categories.cat.categories))]
plt.legend(handles, categories.cat.categories, title=category_column, bbox_to_anchor=(1.05, 1), loc='upper left')

plt.show()

In [ ]:
# Create a color palette
coordinates = adata_cosmx1.obsm['spatial_fov']

category_column = 'slide'
categories = adata_cosmx1.obs[category_column].astype('category')

palette = sns.color_palette('tab20', len(categories.cat.categories))
colors = categories.cat.codes.map(dict(enumerate(palette)))

# Plotting
plt.figure(figsize=(9, 10))
scatter = plt.scatter(coordinates[:, 0], coordinates[:, 1], c=colors, s=0.5)  # Adjust `s` for marker size
plt.title("CD_B4 Spatial Plot for Slide")
plt.xlabel("X coordinate")
plt.ylabel("Y coordinate")

# Create a custom legend
handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=palette[i], markersize=8) 
           for i in range(len(categories.cat.categories))]
plt.legend(handles, categories.cat.categories, title=category_column, bbox_to_anchor=(1.05, 1), loc='upper left')

plt.show()

In [ ]:
# Create a color palette
coordinates = adata_cosmx2.obsm['spatial_fov']

category_column = 'patient'
categories = adata_cosmx2.obs[category_column].astype('category')

palette = sns.color_palette('tab20', len(categories.cat.categories))
colors = categories.cat.codes.map(dict(enumerate(palette)))

# Plotting
plt.figure(figsize=(9, 10))
scatter = plt.scatter(coordinates[:, 0], coordinates[:, 1], c=colors, s=0.5)  # Adjust `s` for marker size
plt.title("CD_B5 Spatial Plot for Patient")
plt.xlabel("X coordinate")
plt.ylabel("Y coordinate")

# Create a custom legend
handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=palette[i], markersize=8) 
           for i in range(len(categories.cat.categories))]
plt.legend(handles, categories.cat.categories, title=category_column, bbox_to_anchor=(1.05, 1), loc='upper left')

plt.show()

In [ ]:
# Create a color palette
coordinates = adata_cosmx2.obsm['spatial_fov']

category_column = 'slide'
categories = adata_cosmx2.obs[category_column].astype('category')

palette = sns.color_palette('tab20', len(categories.cat.categories))
colors = categories.cat.codes.map(dict(enumerate(palette)))

# Plotting
plt.figure(figsize=(9, 10))
scatter = plt.scatter(coordinates[:, 0], coordinates[:, 1], c=colors, s=0.5)  # Adjust `s` for marker size
plt.title("CD_B5 Spatial Plot for Slide")
plt.xlabel("X coordinate")
plt.ylabel("Y coordinate")

# Create a custom legend
handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=palette[i], markersize=8) 
           for i in range(len(categories.cat.categories))]
plt.legend(handles, categories.cat.categories, title=category_column, bbox_to_anchor=(1.05, 1), loc='upper left')

plt.show()

In [ ]:
adata_cosmx.write_h5ad('/path/to/cosmx_data/cd_6k_wtx/whole_trans/Processed_merged/h5ad_obj/norm_anno.h5ad')

In [ ]:
adata_cosmx=sc.read_h5ad('/path/to/cosmx_data/cd_6k_wtx/whole_trans/Processed_merged/h5ad_obj/norm_anno.h5ad')

In [ ]:
adata_cosmx_noninf = adata_cosmx[adata_cosmx.obs['patient'].isin(['UC_NI1','UC_NI2'])]

In [ ]:
adata_cosmx_noninf_df = pd.DataFrame(
    adata_cosmx_noninf.layers['raw'].toarray().astype(int),
    index=adata_cosmx_noninf.obs_names,
    columns=adata_cosmx_noninf.var_names
)

In [ ]:
adata_cosmx_noninf_df.to_csv('/path/to/cosmx_data/cd_6k_wtx/whole_trans/Processed_merged/h5ad_obj/noninf_counts.csv')

In [ ]:
adata_cosmx1.write_h5ad('/path/to/cosmx_data/cd_6k_wtx/whole_trans/Processed_CD_B4/h5ad_obj/norm_anno.h5ad')
adata_cosmx2.write_h5ad('/path/to/cosmx_data/cd_6k_wtx/whole_trans/Processed_CD_B5/h5ad_obj/norm_anno.h5ad')

In [ ]:
adata_cosmx1=sc.read_h5ad('/path/to/cosmx_data/cd_6k_wtx/whole_trans/Processed_CD_B4/h5ad_obj/norm_anno.h5ad')
#adata_cosmx2=sc.read_h5ad('/path/to/cosmx_data/cd_6k_wtx/whole_trans/Processed_CD_B5/h5ad_obj/norm_anno.h5ad')